# Evaluation on validation.csv

In [1]:
import joblib
import pandas as pd
import numpy as np

In [22]:


best_gb = joblib.load('../weights/best_model_rf_tuned.pkl')
scaler  = joblib.load('../weights/scaler.pkl')

selected_features = ['distance', 'quote_signal', 'weight', 'market_index','equip_Dry Van', 'equip_Flatbed', 'equip_Reefer']

In [29]:
import pandas as pd

# Load validation data
val_df = pd.read_csv("../dataset/validation.csv")

# One-hot encode equipment
val_df = pd.get_dummies(
    val_df,
    columns=["equipment"],
    prefix="equip"
)

# Make validation columns exactly match training features
X_val = val_df.reindex(
    columns=selected_features,
    fill_value=0
)

# Apply the scaler fitted on training data
X_val_scaled = scaler.transform(X_val)

print(X_val.shape)
print(X_val.columns.tolist())

(12000, 7)
['distance', 'quote_signal', 'weight', 'market_index', 'equip_Dry Van', 'equip_Flatbed', 'equip_Reefer']


In [34]:
X_val.head(5)

,distance,quote_signal,weight,market_index,equip_Dry Van,equip_Flatbed,equip_Reefer
0,331.4,1.86388,19958.000000,0.90402,False,True,False
1,2406.2,1.86240,34114.000000,0.92851,False,False,True
2,2846.6,1.95480,33435.000000,0.88117,True,False,False
3,2261.6,2.14579,25392.000000,0.89156,True,False,False
4,800.4,2.22541,30506.636248,0.88308,False,True,False


In [30]:
missing = val_df.isnull().sum()
missing_pct = (val_df.isnull().sum() / len(val_df)) * 100

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

print(missing_df)

               missing_count  missing_pct
market_index             249        2.075
weight                   165        1.375
load_id                    0        0.000
pickup_lat                 0        0.000
pickup_lon                 0        0.000
pickup                     0        0.000
delivery                   0        0.000
delivery_lon               0        0.000
delivery_lat               0        0.000
distance                   0        0.000
date                       0        0.000
quote_signal               0        0.000
equip_Dry Van              0        0.000
equip_Flatbed              0        0.000
equip_Reefer               0        0.000


In [31]:
# Market index → average of the same date
val_df["market_index"] = val_df["market_index"].fillna(
    val_df.groupby("date")["market_index"].transform("mean")
)

# Weight → overall dataset average
val_df["weight"] = val_df["weight"].fillna(
    val_df["weight"].mean()
)

# Check missing values
print(val_df[["market_index", "weight"]].isna().sum())

market_index    0
weight          0
dtype: int64


In [32]:
X_val        = val_df[selected_features]


X_val_scaled = scaler.transform(X_val)

val_df['predicted_rate'] = best_gb.predict(X_val_scaled)

# save load_id + predicted_rate only
predictions = val_df[['load_id', 'predicted_rate']]
predictions.to_csv('../results/validation_predictions.csv', index=False)

print("Saved validation_predictions.csv:", predictions.shape)
print(predictions.head())

Saved validation_predictions.csv: (12000, 2)
     load_id  predicted_rate
0  TE-000001      810.911726
1  TE-000002     4698.513624
2  TE-000003     5035.745111
3  TE-000004     4014.963587
4  TE-000005     1802.315997


# Prediction on december_chart_inputs.csv

In [65]:
dec_df = pd.read_csv('../dataset/december-chart-inputs.csv')


In [66]:
dec_df.head(5)

,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,NaN
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,NaN
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,NaN
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,NaN
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,NaN


In [67]:
dec_df.head(5)

,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,NaN
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,NaN
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,NaN
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,NaN
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,NaN


In [68]:
import pandas as pd

# Load files
december = dec_df
validation = val_df

# Make sure date format is consistent
december["date"] = pd.to_datetime(december["date"])
validation["date"] = pd.to_datetime(validation["date"])


# --------------------------------------------------
# 1. Create lookup using pickup + delivery + date
# --------------------------------------------------

validation_lookup = (
    validation
    .groupby(["pickup", "delivery", "date"], as_index=False)
    [["quote_signal", "market_index"]]
    .mean()
)

# Merge into December
december = december.merge(
    validation_lookup,
    on=["pickup", "delivery", "date"],
    how="left"
)


# --------------------------------------------------
# 2. For unmatched rows, use the mean for that DATE
# --------------------------------------------------

date_mean = (
    validation
    .groupby("date")[["quote_signal", "market_index"]]
    .mean()
    .rename(columns={
        "quote_signal": "quote_signal_date_mean",
        "market_index": "market_index_date_mean"
    })
)

# Add date means
december = december.merge(
    date_mean,
    on="date",
    how="left"
)


# --------------------------------------------------
# 3. Fill missing values using date-specific mean
# --------------------------------------------------

december["market_index"] = december["market_index"].fillna(
    december["market_index_date_mean"]
)

december["quote_signal"] = december["quote_signal"].fillna(
    december["quote_signal_date_mean"]
)


# Remove helper columns
december = december.drop(
    columns=["quote_signal_date_mean", "market_index_date_mean"]
)


# --------------------------------------------------
# 4. Save completed file
# --------------------------------------------------

december.to_csv("december_completed.csv", index=False)

# Check
print("Remaining missing values:")
print(december[["quote_signal", "market_index"]].isna().sum())

Remaining missing values:
quote_signal    0
market_index    0
dtype: int64


In [69]:

# One-hot encode equipment
dec_df = pd.get_dummies(
    december,
    columns=['equipment'],
    prefix='equip'
)

# Add missing equipment columns and keep exact training feature order
dec_df = dec_df.reindex(
    columns=selected_features,
    fill_value=False
)

print(dec_df.columns.tolist())

['distance', 'quote_signal', 'weight', 'market_index', 'equip_Dry Van', 'equip_Flatbed', 'equip_Reefer']


In [70]:


# scale & predict
X_dec_scaled = scaler.transform(dec_df)
december['predicted_rate'] = best_gb.predict(X_dec_scaled)

# keep only required columns in exact order
december_output = december[['pickup','delivery','distance','equipment','weight','date','predicted_rate']]
december_output.to_csv('../data/december_chart_inputs.csv', index=False)

print("Saved december_chart_inputs.csv")
print(december_output)

Saved december_chart_inputs.csv
       pickup    delivery  distance equipment  weight       date  \
0   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-01   
1   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-02   
2   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-03   
3   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-04   
4   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-05   
5   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-06   
6   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-07   
7   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-08   
8   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-09   
9   Lexington  Fort Wayne       360   Dry Van   32000 2025-12-10   
10  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-11   
11  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-12   
12  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-13   
13  Lexington  F